# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and analyzing the [FAIR^2 Dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata and print summary
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.

Let's enumerate all record sets present in this dataset and inspect their fields and columns (by unique `@id`).

In [ ]:
# List all record sets with their @id and constituent fields and columns.
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets found. Listing data distributions for file-based record sets:")
    if hasattr(dataset.metadata, 'distribution'):
        for dist in dataset.metadata.distribution:
            print(f"Distribution @id: {getattr(dist, '@id', None)} / Name: {getattr(dist, 'name', None)} / URL: {getattr(dist, 'contentUrl', None)}")
    else:
        print("No file distributions found.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']} / Name: {rs.get('name', None)}")
        # List fields
        if 'field' in rs and rs['field']:
            print("  Fields:")
            for f in rs['field']:
                print(f"    - {f['@id']} / Name: {f.get('name', None)} / DataType: {f.get('dataType', None)}")
        if 'column' in rs and rs['column']:
            print("  Columns:")
            for f in rs['column']:
                print(f"    - {f['@id']} / Name: {f.get('name', None)} / DataType: {f.get('dataType', None)}")

## 3. Data Extraction
Load the main record set as a pandas DataFrame for analysis.

For this dataset, the primary table is included as a file in its distribution, not in an explicit record set array. We'll extract it using its distribution `@id`.

**Note:** You can find all valid distribution `@id`s in the previous overview cell output.

In [ ]:
# We'll use the first available data file as the record set source.
record_set_id = 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034'  # From metadata.distribution[0][@id]
df = pd.DataFrame(dataset.records(record_set=record_set_id))
print(f"Loaded DataFrame shape: {df.shape}")
print(f"Columns (@id):\n{df.columns.tolist()}")
# Show a data sample
df.head(3)

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalizing, and grouping. All field and column references use their `@id` as required.

Let's select a numeric field (e.g. molecular weight, often encoded as `MW` or similar – check the actual columns for correct `@id`).

**Example:** We'll assume one column's `@id` is `MW` for molecular weight, and group by a protein identifier `Accession` if it exists. Adjust the `@id`s based on those found in the actual `df.columns` above.

In [ ]:
# Set the field @id for a numeric analysis (change if needed based on available columns)
numeric_field_id = 'MW' if 'MW' in df.columns else df.select_dtypes('number').columns[0] if len(df.select_dtypes('number').columns) else df.columns[0]
group_field_id = 'Accession' if 'Accession' in df.columns else df.columns[0]

threshold = 20000
try:
    filtered_df = df[df[numeric_field_id].astype(float) > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[numeric_field_id + '_normalized'] = (
        filtered_df[numeric_field_id].astype(float) - filtered_df[numeric_field_id].astype(float).mean()
    ) / filtered_df[numeric_field_id].astype(float).std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        display(grouped_df.head())
except Exception as e:
    print(f"EDA failed: {e}")

## 5. Visualization
Visualize data distributions and relationships between numerical fields using matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field_id].astype(float), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.show()

# If there is another numeric field, show bivariate scatter
num_cols = df.select_dtypes('number').columns
if len(num_cols) > 1:
    plt.figure(figsize=(7, 5))
    xcol, ycol = num_cols[0], num_cols[1]
    sns.scatterplot(data=df, x=xcol, y=ycol)
    plt.title(f"Scatter: {xcol} vs. {ycol}")
    plt.show()

## 6. Conclusion
We successfully used the `mlcroissant` library to parse and analyze the FAIR² dataset:

- Loaded dataset metadata and explored the main data file via its `@id`
- Inspected the available columns (fields) and performed exploratory analysis and filtering
- Visualized distributions of molecular weights and relationships between numerical variables

This approach can be generalized to any dataset with a Croissant schema and used as a baseline for further downstream ML or domain research analysis.